In [ ]:
from py_experimenter.experimenter import PyExperimenter
import pandas as pd

from matplotlib import pyplot as plt
import seaborn as sns


In [ ]:
FIRST_VERSION_DATA_PATH = "plain_data/first_version_table.csv"
FIRST_VERSION_LOG_PATH = "plain_data/first_version_logtable.csv"
UPDATED_DATA_PATH = "plain_data/updated_table.csv"
UPDATED_LOG_PATH = "plain_data/updated_logtable.csv"
REAL_PREDICTION_DATA_PATH = "plain_data/real_prediction_table.csv"
REAL_PREDICTION_LOG_PATH = "plain_data/real_prediction_logtable.csv"


# Get Data

In [36]:
update_data = False

In [37]:
def get_data(table_name:str, update_data) -> [pd.DataFrame, pd.DataFrame]:
    if update_data:
        experimenter = PyExperimenter(experiment_configuration_file_path="conf/experiment_config.yml",
                                database_credential_file_path="conf/database_credentials.yml",
                                table_name=table_name)
        main_table = experimenter.get_table()
        log_table = experimenter.get_logtable("sh_iterations")

        main_table.to_csv(f"plain_data/{table_name}_table.csv", index=False)
        log_table.to_csv(f"plain_data/{table_name}_logtable.csv", index=False)
    else:
        main_table = pd.read_csv(f"plain_data/{table_name}_table.csv")
        log_table = pd.read_csv(f"plain_data/{table_name}_logtable.csv")

    return main_table, log_table


In [38]:
first_version_table, first_version_logtable = get_data("priorbai_experiments", update_data)
kernel_table, kernel_logtable = get_data("priorbai_experiments_lcbench_kernels", update_data)
real_prediction_table, real_prediction_logtable = get_data("priorbai_experiments_lcbench_kernels_ys", update_data)


# Display Head of Main Data

In [39]:
first_version_table.head()

,ID,run_id,num_arms,benchmark,seed,prior,performance_prior_std,sigma0,epsilon,delta,...,T_max,consumed_budget,remaining_arms,num_epsilon_optimal_arms,arm_id_selected,regret,epsilon_optimal,best_arm,end_date,error
0,1,__AUTO__,32,synthetic,0,uniform,0.01,0.01,0.01,0.05,...,32.0,96.0,1.0,1.0,0.0,0.00000,1.0,1.0,2026-01-24 08:41:01,NaN
1,2,__AUTO__,64,synthetic,0,uniform,0.01,0.01,0.01,0.05,...,64.0,224.0,1.0,2.0,0.0,0.00000,1.0,1.0,2026-01-24 08:33:37,NaN
2,3,__AUTO__,128,synthetic,0,uniform,0.01,0.01,0.01,0.05,...,128.0,512.0,1.0,2.0,0.0,0.00000,1.0,1.0,2026-01-24 08:41:00,NaN
3,4,__AUTO__,256,synthetic,0,uniform,0.01,0.01,0.01,0.05,...,256.0,1152.0,1.0,2.0,0.0,0.00000,1.0,1.0,2026-01-24 08:25:16,NaN
4,5,__AUTO__,32,lcbench,0,uniform,0.01,0.01,0.01,0.05,...,52.0,96.0,1.0,3.0,16.0,0.10709,0.0,0.0,2026-01-24 08:37:24,NaN


In [40]:
first_version_logtable.head()

,ID,experiment_id,timestamp,iteration,num_arms,best_arm_included,budget_spent_so_far,N_stop
0,1,7993,2026-01-24 08:22:45,0,32,1,32.0,0.000000e+00
1,2,8354,2026-01-24 08:22:45,0,64,1,64.0,1.377170e+08
2,3,8289,2026-01-24 08:22:45,0,32,1,32.0,5.131354e+07
3,4,6931,2026-01-24 08:22:45,0,128,1,128.0,1.822616e+11
4,5,4451,2026-01-24 08:22:45,0,128,1,128.0,3.585278e+08


# Merge DataFrames

In [41]:
def merge_dataframes(main_df: pd.DataFrame, log_df: pd.DataFrame, ) -> pd.DataFrame:
    return main_df.merge(log_df, left_on="ID", right_on="experiment_id", how="left")

merged_first_version = merge_dataframes(first_version_table, first_version_logtable)
merged_kernel = merge_dataframes(kernel_table, kernel_logtable)
merged_real_prediction = merge_dataframes(real_prediction_table, real_prediction_logtable)

# Plot Synthetic Results

In [ ]:
synthetic_table = merged_first_version[merged_first_version["benchmark"] == "synthetic"]
sns.lineplot(data=synthetic_table, x="sh_iterations", y="best_validation_error", hue="prior_type")